# Pipeline multiagente con CrewAI

Flujo con dos agentes coordinados por CrewAI usando un Qwen2.5-0.5B-Instruct.

In [1]:
%%capture
!pip -q install "crewai==1.14.4" "transformers>=4.45.0" "torch"

In [2]:
import os
from typing import Any, Dict, List, Optional, Union

import torch
from crewai import Agent, BaseLLM, Crew, Process, Task
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
hf_logging.set_verbosity_error()

os.environ.setdefault("HOME", os.getcwd())
os.environ.setdefault("CREWAI_STORAGE_DIR", "crewai_storage")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


class HuggingFaceCrewLLM(BaseLLM):
    def __init__(self, model_name: str = MODEL_NAME, max_new_tokens: int = 180):
        super().__init__(model=model_name, temperature=0)
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, local_files_only=True)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model_obj = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
        self.model_obj.eval()

    def call(
        self,
        messages: Union[str, List[Dict[str, str]]],
        tools: Optional[List[dict]] = None,
        callbacks: Optional[List[Any]] = None,
        available_functions: Optional[Dict[str, Any]] = None,
        **kwargs,
    ) -> str:
        if isinstance(messages, str):
            prompt = messages
        else:
            prompt = "\n".join(str(message.get("content", "")) for message in messages)

        messages_for_model = [
            {"role": "system", "content": "Eres un asistente técnico. Responde en español claro. No repitas instrucciones. Si el tema es RAG, recuerda que significa Retrieval-Augmented Generation."},
            {"role": "user", "content": prompt},
        ]
        prompt = self.tokenizer.apply_chat_template(messages_for_model, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        with torch.no_grad():
            output = self.model_obj.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        generated = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        ).strip()
        if not generated:
            generated = "resultado generado por el modelo local."
        return "Thought: El modelo local genero una respuesta.\nFinal Answer: " + generated

    def supports_function_calling(self) -> bool:
        return False

    def supports_stop_words(self) -> bool:
        return False


llm_local = HuggingFaceCrewLLM()
print("Modelo real cargado:", llm_local.model_name)

/Users/marcechris/Documents/Codex/2026-06-22/a/work/notebook_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7461.89it/s]

Modelo real cargado: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
investigador = Agent(
    role="Investigador de IA aplicada",
    goal="Organizar hallazgos sobre RAG aplicado a soporte al cliente.",
    backstory="Analiza beneficios, riesgos y ejemplos practicos de sistemas de IA.",
    llm=llm_local,
    verbose=True,
    allow_delegation=False,
)

redactor = Agent(
    role="Redactor tecnico",
    goal="Convertir hallazgos tecnicos en una explicacion breve y clara.",
    backstory="Sintetiza informacion tecnica para equipos no especializados.",
    llm=llm_local,
    verbose=True,
    allow_delegation=False,
)

print("Agentes:", investigador.role, "y", redactor.role)

Agentes: Investigador de IA aplicada y Redactor tecnico


In [4]:
tarea_investigacion = Task(
    description="Analiza Retrieval-Augmented Generation (RAG) para mejorar un chatbot de soporte al cliente. Resume beneficios, riesgo y ejemplo practico sin repetir instrucciones.",
    expected_output="Hallazgos con beneficios, riesgo y ejemplo practico.",
    agent=investigador,
)

tarea_redaccion = Task(
    description="Redacta una explicacion breve y final sobre Retrieval-Augmented Generation (RAG), usando los hallazgos previos. No repitas la instruccion.",
    expected_output="Respuesta final breve para compartir con un equipo tecnico.",
    agent=redactor,
    context=[tarea_investigacion],
)

crew = Crew(
    agents=[investigador, redactor],
    tasks=[tarea_investigacion, tarea_redaccion],
    process=Process.sequential,
    verbose=True,
)

resultado = crew.kickoff()
print("\nResultado final:")
print(resultado.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.4                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a83cdf0e-489d-4a4b-adf4-1221cbff428e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analiza Retrieval-Augmented Generation (RAG) para mejorar un chatbot de soporte al cliente. Resume       │
│  beneficios, riesgo y ejemplo practico sin repetir instrucciones.                                               │
│  ID: cda5f590-5a80-45d6-83df-915cbcacdd32                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de IA aplicada                                                                             │
│                                                                                                                 │
│  Task: Analiza Retrieval-Augmented Generation (RAG) para mejorar un chatbot de soporte al cliente. Resume       │
│  beneficios, riesgo y ejemplo practico sin repetir instrucciones.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de IA aplicada                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  En la implementación de la IA en soporte al cliente, como RAG, se ha demostrado que puede mejorar              │
│  significativamente las conversaciones con clientes. Algunos beneficios incluyen:                               │
│                                                                                                                 │
│  Beneficio 1: Mejora la eficiencia del chatbot. Los chatbots pueden procesar más información y responder más    │
│  rápido a los usuarios, lo que reduce el tiempo de respuesta y mejora la satisfacción del cliente.              │
│                                                                                                                 │
│  Beneficio 2: Aumenta la precisión y la calidad de la conversación. Con RAG, los chatbots pueden entender       │
│  mejor las preguntas y respuestas de los usuarios, lo que permite una mayor precisión y calidad de la           │
│  conversación.                                                                                                  │
│                                                                                                                 │
│  Beneficio 3: Mejora la personalización del servicio. Los chatbots pueden adaptarse a las preferencias          │
│  individuales de los usuarios, proporcionando un servicio más personalizado y relev                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor tecnico                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  La Implementación de la IA en Soporte al Cliente (RAG) ha mostrado que puede mejorar significativamente las    │
│  conversaciones con clientes. Este método de generación de texto utiliza datos previamente almacenados para     │
│  generar respuestas a preguntas o solicitudes de usuario.                                                       │
│                                                                                                                 │
│  Beneficio 1: Mejora la eficiencia del chatbot. Los chatbots pueden procesar más información y responder más    │
│  rápidamente a los usuarios, reduciendo el tiempo de respuesta y aumentando la satisfacción del cliente.        │
│                                                                                                                 │
│  Beneficio 2: Aumenta la precisión y la calidad de la conversación. Con RAG, los chatbots pueden entender       │
│  mejor las preguntas y respuestas de los usuarios, permitiendo una mayor precisión y calidad de la              │
│  conversación.                                                                                                  │
│                                                                                                                 │
│  Beneficio 3: Mejora la personalización del servicio. Los chatbots pueden adaptarse a las                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta una explicacion breve y final sobre Retrieval-Augmented Generation (RAG), usando los hallazgos   │
│  previos. No repitas la instruccion.                                                                            │
│  Agent: Redactor tecnico                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a83cdf0e-489d-4a4b-adf4-1221cbff428e                                                                       │
│  Final Output: La Implementación de la IA en Soporte al Cliente (RAG) ha mostrado que puede mejorar             │
│  significativamente las conversaciones con clientes. Este método de generación de texto utiliza datos           │
│  previamente almacenados para generar respuestas a preguntas o solicitudes de usuario.                          │
│                                                                                                                 │
│  Beneficio 1: Mejora la eficiencia del chatbot. Los chatbots pueden procesar más información y responder más    │
│  rápidamente a los usuarios, reduciendo el tiempo de respuesta y aumentando la satisfacción del cliente.        │
│                                                                                                                 │
│  Beneficio 2: Aumenta la precisión y la calidad de la conversación. Con RAG, los chatbots pueden entender       │
│  mejor las preguntas y respuestas de los usuarios, permitiendo una mayor precisión y calidad de la              │
│  conversación.                                                                                                  │
│                                                                                                                 │
│  Beneficio 3: Mejora la personalización del servicio. Los chatbots pueden adaptarse a las                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Resultado final:


La Implementación de la IA en Soporte al Cliente (RAG) ha mostrado que puede mejorar significativamente las conversaciones con clientes. Este método de generación de texto utiliza datos previamente almacenados para generar respuestas a preguntas o solicitudes de usuario. 

Beneficio 1: Mejora la eficiencia del chatbot. Los chatbots pueden procesar más información y responder más rápidamente a los usuarios, reduciendo el tiempo de respuesta y aumentando la satisfacción del cliente.

Beneficio 2: Aumenta la precisión y la calidad de la conversación. Con RAG, los chatbots pueden entender mejor las preguntas y respuestas de los usuarios, permitiendo una mayor precisión y calidad de la conversación.

Beneficio 3: Mejora la personalización del servicio. Los chatbots pueden adaptarse a las
